# 02 - Customer Segmentation (RFM)

This notebook builds an RFM (Recency, Frequency, Monetary) segmentation on the customers identified in notebook 01, compares a quantile-based scoring approach against k-means clustering, and tests whether the resulting segments are stable under two of the arbitrary choices inherited from notebook 01: the cancellation-netting rule and the reference date.

## Research Questions

1. How do customers distribute across Recency, Frequency and Monetary value?
2. Do quantile-based scoring and k-means clustering produce the same segments — and where do they disagree?
3. Are the segments stable if the reference date or the cancellation-netting rule changes?

## Plan

0. Setup & Loading
1. Decision — Cancellation Netting
2. Building the RFM Table
3. RFM Distributions
4. Quantile-Based Segmentation
5. K-Means Clustering
6. Comparing the Two Approaches
7. Stability Test
8. Segment Profiles & Revenue Share
9. Export


# 0. Setup & Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from pathlib import Path
from IPython.display import display, Markdown

PROC = Path("../data/processed")
clean = pd.read_parquet(PROC / "clean_transactions.parquet")

display(Markdown(f"""
### Input
* **Rows:** `{len(clean):,}`
* **Unique customers:** `{clean['Customer ID'].nunique():,}`
* **Date range:** `{clean['InvoiceDate'].min()}` to `{clean['InvoiceDate'].max()}`
"""))


# 1. Decision — Cancellation Netting

Notebook 01 flagged cancellations (`IsCancellation`) but deliberately did not net them out of the shared base, since the right treatment depends on the analysis. Here, for customer monetary value, the decision is:

**Net cancellations into `Monetary`** — a customer's value should reflect what they actually kept, not their gross ordering activity. A cancelled order is not revenue.

**But also keep a separate signal**: `CancellationRate` (share of a customer's invoices that were cancelled), since cancellation behaviour can itself be informative (e.g. for churn in notebook 03) and would otherwise disappear once netted into a single Monetary figure.

# 2. Building the RFM Table

In [ ]:
REFERENCE_DATE = clean["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = clean.groupby("Customer ID").agg(
    Recency=("InvoiceDate", lambda x: (REFERENCE_DATE - x.max()).days),
    Frequency=("Invoice", lambda x: x[~clean.loc[x.index, "IsCancellation"]].nunique()),
    Monetary=("LineRevenue", "sum"),
    TotalInvoices=("Invoice", "nunique"),
    CancelledInvoices=("Invoice", lambda x: x[clean.loc[x.index, "IsCancellation"]].nunique()),
).reset_index()

rfm["CancellationRate"] = (rfm["CancelledInvoices"] / rfm["TotalInvoices"]).fillna(0)

display(Markdown(f"**Reference date:** `{REFERENCE_DATE}`"))
display(Markdown(f"**Customers in RFM table:** `{len(rfm):,}`"))
display(rfm.describe().round(2))


**Results.** 5,880 customers. `Monetary` contains negative values (min = -1,343.24) — customers whose cancellations exceed their net purchases, consistent with the netting decision above. The distribution is heavily right-skewed: Monetary's 75th percentile is £2,214.72 but the max reaches £606,243.25 — consistent with a wholesale-heavy customer base, as already seen in notebook 01. Mean `CancellationRate` is 12%.

# 3. RFM Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["Recency", "Frequency", "Monetary"]):
    sns.histplot(rfm[col], bins=50, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

display(Markdown(f"**Customers with Frequency = 0:** `{(rfm['Frequency'] == 0).sum():,}`"))
display(Markdown(f"**Customers with Monetary <= 0:** `{(rfm['Monetary'] <= 0).sum():,}`"))


**Results.** 26 customers have `Frequency = 0` (all their invoices were cancellations — no real purchase occurred), and 42 have `Monetary <= 0`. The first group is a subset of the second (a customer with only cancellations necessarily has non-positive net revenue). Both groups are excluded from scoring and clustering below, since log-transforms and quantile scoring require strictly positive values, and a customer with no real purchase cannot be meaningfully scored on Frequency or Monetary anyway.

# 4. Quantile-Based Segmentation

Standard RFM scoring: each of R, F, M is split into quintiles (1-5), then scores are combined into a segment label using a common business convention. For `Recency`, lower is better, so quintile 5 = most recent (scored in reverse).

In [ ]:
rfm_scored = rfm[(rfm["Frequency"] > 0) & (rfm["Monetary"] > 0)].copy()

rfm_scored["R_score"] = pd.qcut(rfm_scored["Recency"], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm_scored["F_score"] = pd.qcut(rfm_scored["Frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm_scored["M_score"] = pd.qcut(rfm_scored["Monetary"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm_scored["RFM_score"] = rfm_scored["R_score"].astype(str) + rfm_scored["F_score"].astype(str) + rfm_scored["M_score"].astype(str)

def label_segment(row):
    r, f, m = row["R_score"], row["F_score"], row["M_score"]
    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"
    if r >= 3 and f >= 3:
        return "Loyal Customers"
    if r >= 4 and f <= 2:
        return "New / Promising"
    if r <= 2 and f >= 3:
        return "At Risk"
    if r <= 2 and f <= 2 and m <= 2:
        return "Hibernating"
    return "Needs Attention"

rfm_scored["QuantileSegment"] = rfm_scored.apply(label_segment, axis=1)

display(Markdown("### Quantile segment sizes"))
display(rfm_scored["QuantileSegment"].value_counts())


**Results.** Segment sizes are healthy — no segment is empty or negligibly small: Loyal Customers (1,390), Champions (1,283), Hibernating (1,264), At Risk (830), Needs Attention (620), New/Promising (451). `pd.qcut` did not produce degenerate bins.

# 5. K-Means Clustering

RFM variables are on very different scales (days vs. counts vs. currency) and are right-skewed — both problems that break k-means directly. We log-transform then standardize before clustering.

In [ ]:
X = rfm_scored[["Recency", "Frequency", "Monetary"]].copy()
X_log = np.log1p(X)  # log1p handles Recency=0 safely
X_scaled = StandardScaler().fit_transform(X_log)

inertias, silhouettes = [], []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(K_range), inertias, marker="o")
axes[0].set_title("Elbow method (inertia)")
axes[0].set_xlabel("k")
axes[1].plot(list(K_range), silhouettes, marker="o", color="orange")
axes[1].set_title("Silhouette score")
axes[1].set_xlabel("k")
plt.tight_layout()
plt.show()

display(Markdown("Silhouette scores by k: " + ", ".join(f"k={k}: {s:.3f}" for k, s in zip(K_range, silhouettes))))


**Results.** k=2 gives the clearest separation by a wide margin (silhouette = 0.437), well above the next best, k=4 (0.366). This is the strongest, most unambiguous split in the data.

**Decision — choosing k.** k=2 is kept as the primary, statistically-grounded macro-segmentation, and the quantile-based 6-segment view (section 4) is kept alongside it as an operational refinement: coarser statistical separation, but more actionable for targeting decisions. Both are carried forward rather than picking a single winner (see section 6 for how they cross-validate each other).

In [ ]:
K_CHOSEN = 2

km_final = KMeans(n_clusters=K_CHOSEN, n_init=10, random_state=42)
rfm_scored["Cluster"] = km_final.fit_predict(X_scaled)

cluster_profile = rfm_scored.groupby("Cluster")[["Recency", "Frequency", "Monetary"]].agg(["mean", "median", "count"])
display(Markdown("### Cluster profiles (raw scale)"))
display(cluster_profile.round(1))


**Results.** The two clusters are cleanly interpretable and consistent across all three variables:

- **Cluster 0 — Engaged (2,352 customers, 40%)**: Recency 48.9 days (recently active), Frequency 12.5 orders, Monetary £6,235 mean.
- **Cluster 1 — Disengaged (3,486 customers, 60%)**: Recency 297.6 days (~10 months inactive), Frequency 2.1 orders, Monetary £593 mean.

This confirms the high silhouette score was not a statistical artifact: engagement level (recent + frequent + high-value vs. the opposite) is the single clearest fracture line in the customer base.

# 6. Comparing the Two Approaches

How does the operational segmentation (section 4) distribute across the two macro-clusters?

In [ ]:
crosstab = pd.crosstab(rfm_scored["QuantileSegment"], rfm_scored["Cluster"])
crosstab.columns = ["Cluster 0 (Engaged)", "Cluster 1 (Disengaged)"]

display(Markdown("### Quantile segment x Macro-cluster (k=2)"))
display(crosstab)

ari = adjusted_rand_score(rfm_scored["QuantileSegment"], rfm_scored["Cluster"])
display(Markdown(f"**Adjusted Rand Index:** `{ari:.3f}`"))


**Results.** The two segmentations, built from unrelated methods, agree completely at the extremes: **Champions falls 100% into Engaged (1,283/1,283), Hibernating falls 100% into Disengaged (1,264/1,264)**, and Needs Attention falls 99.8% into Disengaged. This is strong evidence that both segmentations capture a real, not an arbitrary, structure in the data (ARI = 0.227 overall — modest, because the two methods disagree substantially on the middle segments, not because either is wrong).

**Loyal Customers is the one segment that does not resolve cleanly**, splitting 882/508 between Engaged and Disengaged — closer to a genuine mix of two different customer types (recently-active moderate spenders, and long-standing but currently fading high spenders) than a single coherent group from the macro-cluster's perspective. **At Risk** is more coherent (702/830 = 85% Disengaged) but carries a 15% tail in Engaged, worth a closer look before using it operationally without caveat.

# 7. Stability Test

The segmentation depends on two choices made upstream: the reference date and the cancellation-netting rule. We test whether segment assignment is sensitive to each.

In [ ]:
# --- Stability check 1: reference date shifted by -30 days ---
REFERENCE_DATE_ALT = REFERENCE_DATE - pd.Timedelta(30, unit="D")

rfm_alt_date = clean.groupby("Customer ID").agg(
    Recency=("InvoiceDate", lambda x: (REFERENCE_DATE_ALT - x.max()).days),
).reset_index()

recency_shift_corr = rfm_scored.merge(rfm_alt_date, on="Customer ID", suffixes=("", "_alt"))
display(Markdown(f"**Correlation between Recency under the two reference dates:** `{recency_shift_corr['Recency'].corr(recency_shift_corr['Recency_alt']):.4f}`"))

# --- Stability check 2: gross Monetary (cancellations NOT netted) ---
rfm_gross = clean.groupby("Customer ID").agg(
    Monetary_gross=("LineRevenue", lambda x: x[~clean.loc[x.index, "IsCancellation"]].sum())
).reset_index()

compare_monetary = rfm_scored.merge(rfm_gross, on="Customer ID")
pct_changed_decile = (
    pd.qcut(compare_monetary["Monetary"], 10, labels=False, duplicates="drop") !=
    pd.qcut(compare_monetary["Monetary_gross"], 10, labels=False, duplicates="drop")
).mean() * 100

display(Markdown(f"**% of customers whose Monetary decile changes if cancellations are excluded entirely rather than netted:** `{pct_changed_decile:.1f}%`"))


**Results.**
- **Reference date**: correlation = 1.0000 — Recency ranking is fully stable to a 30-day shift, as expected.
- **Cancellation netting**: 5.9% of customers change Monetary decile depending on whether cancellations are netted or excluded entirely. A real but modest effect — this decision matters for roughly 1 in 17 customers, not the majority.

# 8. Segment Profiles & Revenue Share

In [ ]:
# Macro-level revenue concentration (k=2)
macro_summary = rfm_scored.groupby("Cluster").agg(
    n_customers=("Customer ID", "count"),
    total_monetary=("Monetary", "sum"),
).reset_index()
macro_summary["Cluster"] = macro_summary["Cluster"].map({0: "Engaged", 1: "Disengaged"})
macro_summary["pct_customers"] = macro_summary["n_customers"] / macro_summary["n_customers"].sum() * 100
macro_summary["pct_revenue"] = macro_summary["total_monetary"] / macro_summary["total_monetary"].sum() * 100

display(Markdown("### Macro-level revenue concentration (k=2)"))
display(macro_summary.round(2))

# Operational-level (quantile, 6 segments)
segment_summary = rfm_scored.groupby("QuantileSegment").agg(
    n_customers=("Customer ID", "count"),
    avg_recency=("Recency", "mean"),
    avg_frequency=("Frequency", "mean"),
    total_monetary=("Monetary", "sum"),
).reset_index()
segment_summary["pct_customers"] = segment_summary["n_customers"] / segment_summary["n_customers"].sum() * 100
segment_summary["pct_revenue"] = segment_summary["total_monetary"] / segment_summary["total_monetary"].sum() * 100

display(Markdown("### Operational segment profiles (quantile, 6 segments)"))
display(segment_summary.round(2).sort_values("pct_revenue", ascending=False))


## Cross-validation of the two segmentation levels

The two segmentation approaches — quantile-based (6 operational segments) and k-means (2 macro-clusters) — agree completely at the extremes: **Champions falls 100% into the Engaged cluster, Hibernating falls 100% into Disengaged**, despite being built from unrelated methods. This is strong evidence that both segmentations capture a real, not an arbitrary, structure in the data.

The one segment that does not resolve cleanly is **Loyal Customers**, which splits 882/508 between Engaged and Disengaged — nearly a genuine mix rather than a coherent group from the macro-cluster's perspective. This likely reflects two different customer types converging on similar quantile scores: recently active moderate spenders, and long-standing but currently fading high spenders. **At Risk** is more coherent (85% Disengaged) but still carries a 15% tail in Engaged, worth a closer look before using it operationally.

**Revenue concentration is even sharper at the macro level**: the Engaged cluster (40.3% of customers) generates **87.6%** of total revenue — more concentrated than the quantile view's Champions-only figure (22% of customers → 69% of revenue), since the macro-cluster also captures high-value Loyal Customers that quantile scoring separates out.

**Headline finding for this notebook**: whichever granularity is used, a minority of customers drives the large majority of revenue — 22% → 69% at the operational level, 40% → 88% at the macro level. Both cuts are kept because they answer different questions: the macro-cluster is the cleanest possible engagement signal, the operational segments are what a targeting campaign would actually act on.

# 9. Export

Save the segment table for the dashboard and for notebooks 03/04, which will use `Customer ID` as the join key.

In [ ]:
out_cols = ["Customer ID", "Recency", "Frequency", "Monetary", "CancellationRate", "QuantileSegment", "Cluster"]
segment_export = rfm_scored[out_cols].rename(columns={"QuantileSegment": "Segment", "Cluster": "MacroCluster"})
segment_export["MacroCluster"] = segment_export["MacroCluster"].map({0: "Engaged", 1: "Disengaged"})
segment_export.to_parquet(PROC / "customer_segments.parquet", index=False)

display(Markdown(f"Saved `{len(segment_export):,}` customer segments to `customer_segments.parquet`"))
